# Probability for Engineers
# Chapter 4 – Discrete Random Variables

---

This notebook covers three parts of Chapter 4:

| Part | Topics |
|------|--------|
| **Part 1** | Random variables, CDF, PMF |
| **Part 2** | Expected value, Variance, Bernoulli & Binomial |
| **Part 3** | Poisson, Geometric, Negative Binomial |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.special import comb, factorial

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
np.random.seed(42)
print("Libraries loaded successfully ✓")

---
## Part 1 – Random Variables, CDF, and PMF

### 1.1 What Is a Random Variable?

> **Definition:** A Random Variable $X$ is a real-valued function defined on a sample space $S$: $X : S \\to \\mathbb{R}$

**Example:** Rolling two fair six-sided dice – $S = \\{(a_1,a_2):1\\le a_1,a_2\\le6\\}$

$$X(a)=a_1, \\quad Y(a)=|a_1-a_2|, \\quad Z(a)=a_1+a_2$$

In [ ]:
sample_space = [(a1,a2) for a1 in range(1,7) for a2 in range(1,7)]
X_vals = [a[0] for a in sample_space]
Y_vals = [abs(a[0]-a[1]) for a in sample_space]
Z_vals = [a[0]+a[1] for a in sample_space]

print("Sample space size:", len(sample_space))
print("X unique values:", sorted(set(X_vals)))
print("Y unique values:", sorted(set(Y_vals)))
print("Z unique values:", sorted(set(Z_vals)))

p_X_1 = X_vals.count(1) / len(X_vals)
print("P(X=1) =", round(p_X_1, 4), " (theoretical: 1/6 ≈", round(1/6, 4), ")")

### 1.2 Discrete Random Variable and PMF

> **Definition:** A random variable that can take at most a **countable** number of values is called a *discrete random variable*.

**Probability Mass Function (PMF):** $p(x) = P(X = x)$

**PMF conditions:** $p(x) \\ge 0$ and $\\displaystyle\\sum_{x} p(x) = 1$

In [ ]:
x_vals = np.array([1, 2, 3, 4])
p_vals = np.array([1/4, 1/2, 1/8, 1/8])

print("PMF Table:")
print("-" * 25)
for x, p in zip(x_vals, p_vals):
    print(f"  p({x}) = {p:.4f}")
print("-" * 25)
print(f"Total = {p_vals.sum():.4f}")
print("Valid PMF?", "Yes ✓" if np.isclose(p_vals.sum(), 1) else "No ✗")

### 1.3 Cumulative Distribution Function (CDF)

> **Definition:** $F(x) = P(X \\le x),\\quad -\\infty < x < \\infty$

**Properties:**
1. $F(x)$ is non-decreasing, 2. $\\lim_{x\\to\\infty}F(x)=1$, 3. $\\lim_{x\\to-\\infty}F(x)=0$, 4. Right-continuous

$$P(a < X \\le b) = F(b) - F(a)$$

In [ ]:
x_vals = np.array([1, 2, 3, 4])
p_vals = np.array([1/4, 1/2, 1/8, 1/8])
F_vals = np.cumsum(p_vals)

print("CDF Table:")
intervals = ['a < 1', '1<=a<2', '2<=a<3', '3<=a<4', '4<=a']
F_display = np.concatenate([[0], F_vals])
for interval, f in zip(intervals, F_display):
    print(f"  F({interval}) = {f:.4f}")

print("Example calculations:")
print("  P(X<=2) =", F_vals[1])
print("  P(X>2)  =", round(1 - F_vals[1], 4))
print("  P(1<X<=3) = F(3)-F(1) =", round(F_vals[2]-F_vals[0], 4))
print("  P(X=2) = F(2)-F(1-) =", round(F_vals[1]-F_vals[0], 4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.bar(x_vals, p_vals, width=0.4, color='steelblue', alpha=0.75, edgecolor='navy')
ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.set_title('Probability Mass Function (PMF)')
ax.set_xticks(x_vals)
for x, p in zip(x_vals, p_vals):
    ax.text(x, p+0.005, f'{p:.3f}', ha='center', fontsize=9)

ax = axes[1]
x_plot = np.array([0,1,1,2,2,3,3,4,4,5])
F_plot  = np.array([0,0,1/4,1/4,3/4,3/4,7/8,7/8,1,1])
ax.step(x_plot, F_plot, where='post', color='crimson', linewidth=2)
ax.scatter(x_vals, F_vals, s=60, color='crimson', zorder=5)
ax.scatter(x_vals, np.concatenate([[0], F_vals[:-1]]),
           s=60, facecolors='white', edgecolors='crimson', zorder=5)
ax.set_xlabel('x'); ax.set_ylabel('F(x)')
ax.set_title('Cumulative Distribution Function (CDF)')
ax.set_ylim(-0.05, 1.1)
ax.axhline(1, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout(); plt.show()

### 1.4 Coin Tossing (Geometric Series Example)

Toss a coin until heads appears. Probability of heads on each toss is $p$.

$$P(X=n) = (1-p)^{n-1}p \\qquad F(n)=1-(1-p)^n$$

In [ ]:
p = 0.5
n_vals   = np.arange(1, 11)
pmf_vals = (1-p)**(n_vals-1) * p
cdf_vals = 1 - (1-p)**n_vals

print(f"Coin Toss PMF/CDF (p={p})")
print("-"*45)
h_n, h_pmf, h_cdf = "n", "P(X=n)", "F(n)"
print(f"  {h_n:>4} | {h_pmf:>10} | {h_cdf:>10}")
print("-"*45)
for n, pmf, cdf in zip(n_vals, pmf_vals, cdf_vals):
    print(f"  {n:>4} | {pmf:>10.4f} | {cdf:>10.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(n_vals, pmf_vals, color='teal', alpha=0.7, edgecolor='darkslategray')
axes[0].set_title(f'Coin Toss PMF  (p={p})')
axes[0].set_xlabel('n (number of tosses)'); axes[0].set_ylabel('P(X=n)')
axes[0].set_xticks(n_vals)

axes[1].step(n_vals, cdf_vals, where='post', color='darkorange', linewidth=2)
axes[1].scatter(n_vals, cdf_vals, s=50, color='darkorange')
axes[1].axhline(1, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Coin Toss CDF  F(n)=1-(1-p)^n')
axes[1].set_xlabel('n'); axes[1].set_ylabel('F(n)')
axes[1].set_xticks(n_vals)
plt.tight_layout(); plt.show()

---
## Part 2 – Expected Value, Variance, Bernoulli, and Binomial

### 2.1 Expected Value

> $$E[X] = \\sum_{x: p(x)>0} x \\cdot p(x)$$

**Important properties:**
- $E[g(X)] = \\sum_x g(x)\\cdot p(x)$
- $E[aX+b] = aE[X]+b$ (linearity)
- $E[X+Y] = E[X]+E[Y]$

In [ ]:
# Example 1: Fair 4-sided die
x = np.array([1, 2, 3, 4])
p = np.ones(4) / 4
E_X = np.sum(x * p)
print("Fair 4-sided die")
print("  E[X] =", E_X)

# Example 2: p(1)=1/4, p(2)=1/2, p(3)=1/8, p(4)=1/8
x2 = np.array([1, 2, 3, 4])
p2 = np.array([1/4, 1/2, 1/8, 1/8])
E_X2 = np.sum(x2 * p2)
print("Custom distribution")
print("  E[X] =", round(E_X2, 4))

# Linear transformation
a, b = 3, 2
E_direct = np.sum((a*x + b) * p)
E_rule   = a * E_X + b
print("Linearity rule E[3X+2]:")
print("  Direct  :", E_direct)
print("  3*E[X]+2:", E_rule)
print("  Equal?  ", "✓" if np.isclose(E_direct, E_rule) else "✗")

# E[X^2] example
x3 = np.array([-1, 0, 1])
p3 = np.array([1/4, 1/2, 1/4])
E_Xsq = np.sum(x3**2 * p3)
print("For X in {-1,0,1}: E[X^2] =", E_Xsq)

### 2.2 Variance and Standard Deviation

$$\\sigma^2 = \\text{Var}(X) = E[(X-\\mu)^2] = E[X^2] - \\mu^2$$

$$\\sigma = \\text{SD}(X) = \\sqrt{\\text{Var}(X)}$$

**Properties:** $\\text{Var}(aX+b) = a^2\\text{Var}(X)$, $\\text{Var}(b)=0$

In [ ]:
x = np.array([1, 2, 3, 4])
p = np.ones(4) / 4

mu   = np.sum(x * p)
E_X2 = np.sum(x**2 * p)
var  = E_X2 - mu**2
sd   = np.sqrt(var)

print("Fair 4-sided die:")
print("  E[X]   =", mu)
print("  E[X^2] =", E_X2)
print("  Var(X) =", var)
print("  SD(X)  =", round(sd, 4))

# X1 vs X2 – same mean, different spread
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vals, color, lbl in zip(
    axes,
    [[-1,1], [-2,2]],
    ['steelblue', 'tomato'],
    ['X1: sigma=1', 'X2: sigma=2']
):
    probs = [0.5, 0.5]
    ax.bar(vals, probs, width=0.3, color=color, alpha=0.75, edgecolor='black')
    mu_i  = np.sum(np.array(vals)*np.array(probs))
    var_i = np.sum(np.array(vals)**2 * np.array(probs)) - mu_i**2
    ax.set_title(f'{lbl}  mu={mu_i}, sigma^2={var_i}')
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(0, 0.65)
    ax.set_xlabel('x'); ax.set_ylabel('p(x)')
    ax.axvline(mu_i, color='black', linestyle='--', linewidth=1.2, label='mu')
    ax.legend()
plt.suptitle('Same Mean – Different Spread', fontsize=13)
plt.tight_layout(); plt.show()

a = 3
print("Var(3X)   = 3^2 * Var(X) =", a**2, "*", var, "=", a**2 * var)
print("Var(X+5)  = Var(X)       =", var, " (adding a constant does not change variance)")

### 2.3 Bernoulli Distribution

> $X \\sim \\text{Ber}(p)$: $P(X=1)=p$, $P(X=0)=1-p$

$$E[X]=p, \\qquad \\text{Var}(X)=p(1-p)$$

**Examples:** Fair coin → $p=0.5$ | Rolling a 6 → $p=1/6$

In [ ]:
p_values = [0.2, 0.5, 0.8]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, p in zip(axes, p_values):
    probs = [1-p, p]
    bars = ax.bar([0,1], probs, color=['salmon','steelblue'],
                  alpha=0.8, edgecolor='black', width=0.4)
    ax.set_xticks([0,1])
    ax.set_xticklabels(['0 (Failure)', '1 (Success)'])
    ax.set_ylim(0, 1.1); ax.set_ylabel('p(x)')
    ax.set_title(f'Ber(p={p})  mu={p}, sigma^2={round(p*(1-p),4)}')
    for bar, val in zip(bars, probs):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.02,
                f'{val:.2f}', ha='center', fontsize=10)

plt.suptitle('Bernoulli Distribution', fontsize=13)
plt.tight_layout(); plt.show()

p = 0.7
print("Bernoulli(p=0.7):")
print("  E[X]   =", p)
print("  Var(X) =", round(p*(1-p), 4))
print("  Verification: E[X^2]-E[X]^2 =", round(p - p**2, 4))

### 2.4 Binomial Distribution

> $X \\sim \\text{Bin}(n,p)$: number of successes in $n$ independent trials

$$p(k)=\\binom{n}{k}p^k(1-p)^{n-k}, \\quad k=0,1,\\ldots,n$$

$$E[X]=np, \\qquad \\text{Var}(X)=np(1-p)$$

**Conditions:** Independent | Fixed $n$ | Binary outcome | Fixed $p$

**Binomial Theorem:** $\\sum_{k=0}^{n}\\binom{n}{k}p^k(1-p)^{n-k} = [p+(1-p)]^n = 1$ ✓

In [ ]:
def binom_pmf(n, p, k):
    return float(comb(n, k, exact=True)) * p**k * (1-p)**(n-k)

# 4 dice, p=1/6 example
n, p = 4, 1/6
k_vals = np.arange(0, n+1)
pmf = np.array([binom_pmf(n, p, k) for k in k_vals])

print(f"Bin(n={n}, p=1/6) – roll 4 dice, count 6s:")
print("-"*40)
for k, prob in zip(k_vals, pmf):
    print(f"  P(X={k}) = {prob:.6f}")
print(f"  Total    = {pmf.sum():.6f}  ✓")
print(f"  E[X]     = np = {n}*{1/6:.4f} = {n*1/6:.4f}")
print(f"  Var(X)   = np(1-p) = {n*(1/6)*(5/6):.4f}")

In [ ]:
# Binomial PMF – four parameter combinations
configs = [
    (4,  1/6, 'Bin(4,1/6)',  'steelblue'),
    (20, 1/6, 'Bin(20,1/6)', 'salmon'),
    (4,  0.5, 'Bin(4,1/2)',  'seagreen'),
    (20, 0.5, 'Bin(20,1/2)', 'darkorange'),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (n, p, title, color) in zip(axes.flat, configs):
    k = np.arange(0, n+1)
    pmf = stats.binom.pmf(k, n, p)
    ax.bar(k, pmf, color=color, alpha=0.7, edgecolor='white')
    ax.set_title(f'{title}  mu={round(n*p,2)}, sigma^2={round(n*p*(1-p),2)}')
    ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
    ax.axvline(n*p, color='black', linestyle='--', linewidth=1.5, label=f'mu={round(n*p,2)}')
    ax.legend(fontsize=9)
plt.suptitle('Binomial Distribution PMF', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Real-world example: part-time employment
n, p, k_target = 10, 0.42, 8
prob = stats.binom.pmf(k_target, n, p)
print(f"42% of the class works part-time – probability that exactly {k_target}")
print(f"out of n={n} students work part-time:")
print(f"  P(X={k_target}) = C(10,8) x 0.42^8 x 0.58^2 = {prob:.6f}")

print("\nCumulative probabilities:")
for k in range(n+1):
    bar = chr(9608) * int(stats.binom.pmf(k, n, p) * 200)
    print(f"  P(X={k:2d}) = {stats.binom.pmf(k,n,p):.4f}  {bar}")

print(f"\n  P(X<=5) = {stats.binom.cdf(5,n,p):.4f}")
print(f"  P(X>5)  = {1-stats.binom.cdf(5,n,p):.4f}")

---
## Part 3 – Poisson, Geometric, and Negative Binomial Distributions

### 3.1 Poisson Distribution

> $X \\sim P(\\lambda)$:
> $$p(k) = e^{-\\lambda}\\frac{\\lambda^k}{k!}, \\quad k=0,1,2,\\ldots$$

$$E[X] = \\lambda, \\qquad \\text{Var}(X) = \\lambda$$

**Use cases:** Number of events per unit time/area (failures, calls, yeast cells...)

In [ ]:
lambdas = [0.5, 1, 2, 8]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colors = ['steelblue', 'salmon', 'seagreen', 'darkorange']

for ax, lam, color in zip(axes.flat, lambdas, colors):
    k = np.arange(0, 21)
    pmf = stats.poisson.pmf(k, lam)
    ax.bar(k, pmf, color=color, alpha=0.7, edgecolor='white')
    ax.set_title(f'Pois(lambda={lam})  mu=sigma^2={lam}')
    ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
    ax.axvline(lam, color='black', linestyle='--', linewidth=1.5, label=f'lambda={lam}')
    ax.legend(fontsize=9)

plt.suptitle('Poisson Distribution PMF', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# PMF validity: verification via Taylor Series
print("Poisson PMF sum = 1 verification:")
for lam in [0.5, 1, 2, 8]:
    # range(50) is sufficient as each term converges to ~0 from k=50 onward
    total = sum(np.exp(-lam) * lam**k / factorial(k) for k in range(50))
    print(f"  lambda={lam}  ->  Total = {total:.10f}  {chr(10003)}")

# Vassar example
print("\nVassar Example:")
lam = 1.73
p0 = stats.poisson.pmf(0, lam)
p1 = stats.poisson.pmf(1, lam)
print(f"  X ~ Pois(lambda={lam})")
print(f"  p(0) = {p0:.4f}")
print(f"  p(1) = {p1:.4f}")
print(f"  P(X<=1) = {p0+p1:.4f}")

### 3.2 Poisson Approximation to the Binomial

For $X \\sim \\text{Bin}(n,p)$ with small $p$, large $n$, and moderate $\\lambda=np$:

$$\\text{Bin}(n,p) \\approx \\text{Pois}(\\lambda=np)$$

In [ ]:
n, p = 1000, 0.05
lam = n * p
k = np.arange(20, 85)  # covers approx +-4*sigma ~ [22,78] for lambda=50
pmf_binom   = stats.binom.pmf(k, n, p)
pmf_poisson = stats.poisson.pmf(k, lam)

plt.figure(figsize=(12, 5))
plt.bar(k, pmf_binom, width=0.7, label=f'Bin(n={n},p={p})', alpha=0.6, color='steelblue')
plt.plot(k, pmf_poisson, 'o-', color='red', markersize=4,
         label=f'Pois(lambda={lam})', linewidth=1.5)
plt.xlabel('k'); plt.ylabel('Probability')
plt.title(f'Binomial–Poisson Approximation  (n={n}, p={p}, lambda={lam})')
plt.legend(); plt.tight_layout(); plt.show()

max_error = np.max(np.abs(pmf_binom - pmf_poisson))
print(f"Maximum absolute error: {max_error:.6f}  -> approximation is very good ✓")

### 3.3 Geometric Distribution

> $X \\sim \\text{Geom}(p)$: number of trials until the first success

$$p(k) = (1-p)^{k-1}p, \\quad k=1,2,\\ldots$$

$$E[X]=\\frac{1}{p}, \\quad \\text{Var}(X)=\\frac{1-p}{p^2}, \\quad P(X\\le k)=1-(1-p)^k$$

**Memoryless property:** $P(X>n+k\\mid X>n) = P(X>k)$

In [ ]:
p_vals = [0.2, 0.5, 0.8]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
colors = ['sandybrown', 'cadetblue', 'mediumpurple']

for ax, p, color in zip(axes, p_vals, colors):
    k = np.arange(1, 21)
    pmf = stats.geom.pmf(k, p)
    ax.bar(k, pmf, color=color, alpha=0.75, edgecolor='white')
    mu_g  = 1/p
    var_g = (1-p)/p**2
    ax.set_title(f'Geom(p={p})  mu={mu_g:.2f}, sigma^2={var_g:.2f}')
    ax.set_xlabel('k'); ax.set_ylabel('P(X=k)')
    ax.axvline(mu_g, color='black', linestyle='--', linewidth=1.5)

plt.suptitle('Geometric Distribution PMF', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Memoryless property verification
p = 0.3
n, k = 5, 3
P_gt_nk = (1-p)**(n+k)
P_gt_n  = (1-p)**n
P_cond  = P_gt_nk / P_gt_n
P_gt_k  = (1-p)**k

print("Memoryless property verification (p=0.3, n=5, k=3):")
print(f"  P(X>{n+k})        = (1-p)^{n+k} = {P_gt_nk:.6f}")
print(f"  P(X>{n})          = (1-p)^{n}   = {P_gt_n:.6f}")
print(f"  P(X>{n+k}|X>{n})  = {P_cond:.6f}")
print(f"  P(X>{k})          = (1-p)^{k}   = {P_gt_k:.6f}")
print(f"  Equal? {chr(10003)+' Memoryless property confirmed!' if np.isclose(P_cond, P_gt_k) else chr(10007)}")

# Gambler's fallacy
rounds = np.arange(1, 16)
p_roulette = 18/38
pmf_r = stats.geom.pmf(rounds, p_roulette)
colors_r = ['tomato' if t <= 5 else 'steelblue' for t in rounds]

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(rounds, pmf_r, color=colors_r, alpha=0.8, edgecolor='white')
patch1 = mpatches.Patch(color='tomato',    alpha=0.8, label='Losses (rounds 1-5)')
patch2 = mpatches.Patch(color='steelblue', alpha=0.8, label='Subsequent rounds')
ax.legend(handles=[patch1, patch2])
ax.set_xlabel('Round'); ax.set_ylabel('P(X=k)')
ax.set_title(f'Roulette Geometric(p={p_roulette:.3f}) – Gambler\'s Fallacy\n'
             'Probability of winning after 5 consecutive losses is UNCHANGED!')
ax.set_xticks(rounds)
plt.tight_layout(); plt.show()

print(f"p_roulette = 18/38 = {p_roulette:.4f}  (probability of winning on any single spin is constant)")

### 3.4 Negative Binomial Distribution

> $X \\sim \\text{NB}(r,p)$: number of trials until the $r$-th success

$$p(k)=\\binom{k-1}{r-1}(1-p)^{k-r}p^r, \\quad k=r,r+1,\\ldots$$

$$E[X]=\\frac{r}{p}, \\qquad \\text{Var}(X)=\\frac{r(1-p)}{p^2}$$

**Relationship:** $\\text{Geometric}(p) = \\text{NB}(1,p)$

In [ ]:
configs = [
    (2, 0.2, 'NB(r=2,p=0.2)', 'sandybrown'),
    (2, 0.5, 'NB(r=2,p=0.5)', 'cadetblue'),
    (3, 0.2, 'NB(r=3,p=0.2)', 'mediumpurple'),
    (3, 0.5, 'NB(r=3,p=0.5)', 'seagreen'),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, (r, p, title, color) in zip(axes.flat, configs):
    k_range = np.arange(r, r+21)
    pmf_nb = np.array([
        float(comb(k-1, r-1, exact=True)) * (1-p)**(k-r) * p**r
        for k in k_range
    ])
    ax.bar(k_range, pmf_nb, color=color, alpha=0.75, edgecolor='white')
    mu_nb  = r / p
    var_nb = r * (1-p) / p**2
    ax.set_title(f'{title}  mu={mu_nb:.2f}, sigma^2={var_nb:.2f}')
    ax.set_xlabel('k (total trials)'); ax.set_ylabel('P(X=k)')
    ax.axvline(mu_nb, color='black', linestyle='--', linewidth=1.5)

plt.suptitle('Negative Binomial Distribution PMF', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Verification of relationship to Geometric
p = 0.4
k = np.arange(1, 15)
pmf_geom = stats.geom.pmf(k, p)
pmf_nb1  = np.array([float(comb(ki-1,0,exact=True))*(1-p)**(ki-1)*p for ki in k])
print("NB(r=1,p) = Geometric(p) verification:")
print(f"  Max difference: {np.max(np.abs(pmf_geom - pmf_nb1)):.2e}  {chr(10003)}")

---
## Chapter 4 Summary – Discrete Distributions

| Distribution | Notation | Support | $E[X]$ | $\\text{Var}(X)$ |
|-------------|----------|---------|--------|------------------|
| Bernoulli | $\\text{Ber}(p)$ | $\\{0,1\\}$ | $p$ | $p(1-p)$ |
| Binomial | $\\text{Bin}(n,p)$ | $\\{0,...,n\\}$ | $np$ | $np(1-p)$ |
| Poisson | $\\text{Pois}(\\lambda)$ | $\\{0,1,2,...\\}$ | $\\lambda$ | $\\lambda$ |
| Geometric | $\\text{Geom}(p)$ | $\\{1,2,...\\}$ | $1/p$ | $(1-p)/p^2$ |
| Neg. Binomial | $\\text{NB}(r,p)$ | $\\{r,r+1,...\\}$ | $r/p$ | $r(1-p)/p^2$ |

In [ ]:
print("=" * 65)
print(f"  {'Distribution':<22}  {'Mean':>10}  {'Variance':>10}  {'SD':>10}")
print("=" * 65)

rows = [
    ('Ber(p=0.6)',          0.6,       0.6*0.4),
    ('Bin(n=10, p=0.6)',    10*0.6,    10*0.6*0.4),
    ('Pois(lambda=3)',       3,         3),
    ('Geom(p=0.4)',          1/0.4,    (1-0.4)/0.4**2),
    ('NB(r=3, p=0.4)',       3/0.4,    3*(1-0.4)/0.4**2),
]
for name, mu, var in rows:
    print(f"  {name:<22}  {mu:>10.4f}  {var:>10.4f}  {np.sqrt(var):>10.4f}")
print("=" * 65)
print("\nChapter 4 complete! ✓")

In [ ]:
# All distributions – overview plot
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Bernoulli
p = 0.6
ax = axes[0,0]
ax.bar([0,1],[1-p,p], color=['salmon','steelblue'], alpha=0.75, width=0.4)
ax.set_xticks([0,1]); ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.set_title(f'Bernoulli(p={p})  mu={p}')

# Binomial
n, p = 10, 0.6
ax = axes[0,1]
k = np.arange(0, n+1)
ax.bar(k, stats.binom.pmf(k,n,p), color='steelblue', alpha=0.7)
ax.axvline(n*p, color='red', linestyle='--', label=f'mu={n*p}')
ax.set_title(f'Bin(n={n},p={p})')
ax.set_xlabel('k'); ax.legend(fontsize=8)

# Poisson
lam = 3
ax = axes[0,2]
k = np.arange(0, 15)
ax.bar(k, stats.poisson.pmf(k,lam), color='seagreen', alpha=0.7)
ax.axvline(lam, color='red', linestyle='--', label=f'lambda={lam}')
ax.set_title(f'Poisson(lambda={lam})')
ax.set_xlabel('k'); ax.legend(fontsize=8)

# Geometric
p = 0.4
ax = axes[1,0]
k = np.arange(1, 20)
ax.bar(k, stats.geom.pmf(k,p), color='darkorange', alpha=0.7)
ax.axvline(1/p, color='red', linestyle='--', label=f'mu={1/p:.2f}')
ax.set_title(f'Geometric(p={p})')
ax.set_xlabel('k'); ax.legend(fontsize=8)

# NB r=2
r, p = 2, 0.4
ax = axes[1,1]
k_r = np.arange(r, r+20)
pmf_nb = np.array([float(comb(k-1,r-1,exact=True))*(1-p)**(k-r)*p**r for k in k_r])
ax.bar(k_r, pmf_nb, color='mediumpurple', alpha=0.7)
ax.axvline(r/p, color='red', linestyle='--', label=f'mu={r/p:.2f}')
ax.set_title(f'NB(r={r},p={p})')
ax.set_xlabel('k'); ax.legend(fontsize=8)

# NB r=5
r, p = 5, 0.4
ax = axes[1,2]
k_r = np.arange(r, r+30)
pmf_nb = np.array([float(comb(k-1,r-1,exact=True))*(1-p)**(k-r)*p**r for k in k_r])
ax.bar(k_r, pmf_nb, color='cadetblue', alpha=0.7)
ax.axvline(r/p, color='red', linestyle='--', label=f'mu={r/p:.2f}')
ax.set_title(f'NB(r={r},p={p})')
ax.set_xlabel('k'); ax.legend(fontsize=8)

plt.suptitle('Chapter 4 – Discrete Distributions Overview', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()